Code for Analysis 1: Neural Connectome Reconstruction

--> Part 1: Differential Gene Expression Reconstruction 

--> Part 2: Regulatory Gene Expression Reconstruction 

In [2]:
# import necessary libraries
import random
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.cluster import SpectralClustering
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# file configurations
INPUT_DIR = Path("inputs")
REGULON_FILES = {                      
    "TF Link":   "../data/TFLink_Caenorhabditis_elegans_interactions_All_GMT_proteinName_v1.0.gmt",
}
SEED = 0
TYPES = ["sensory", "interneuron", "motor", "pharyngeal"]
OUT = Path("../figures/")
XS = [5, 10, 25, 50, 100, 250, 500, 1000, 2000]   # top-X DE genes per neuron (sensitivity analysis)
N_PERM = 100                           # edge-swap permutations
OUT.mkdir(exist_ok=True)

In [3]:
# setup / helper functions
def cpm(counts):
    return counts / counts.sum(axis=0) * 1e6


def one_vs_all_log2fc(counts, pseudo=1.0):
    """log2 fold change of each column (CPM) vs the mean of all other columns."""
    c = cpm(counts)
    n = c.shape[1]
    rest = (c.sum(axis=1).values[:, None] - c.values) / (n - 1)
    return pd.DataFrame(np.log2((c.values + pseudo) / (rest + pseudo)),
                        index=c.index, columns=c.columns)


def build_connectome(edges, meta_types):
    G = nx.Graph()
    G.add_nodes_from(meta_types.index)
    for r in edges.itertuples(index=False):
        a, b = r[0], r[1]
        w = r[2] if len(r) > 2 else 1.0
        if a in G and b in G and a != b:
            if G.has_edge(a, b):
                G[a][b]["weight"] += w
            else:
                G.add_edge(a, b, weight=w)
    nx.set_node_attributes(G, meta_types.to_dict(), "type")
    return G


def read_gene_sets(path):
    p, sets = Path(path), {}
    if p.suffix.lower() == ".gmt":
        for line in open(p):
            f = line.rstrip("\n").split("\t")
            if len(f) >= 3:
                sets[f[0]] = {g for g in f[2:] if g}
    else:
        df = pd.read_csv(p)
        tf, tg = df.columns[:2]
        for name, sub in df.groupby(tf):
            sets[str(name)] = set(sub[tg].astype(str)) | {str(name)}
    return sets

In [4]:
# data load

meta = pd.read_csv(INPUT_DIR / "neurons.csv")
counts = pd.read_csv(INPUT_DIR / "counts.csv", index_col=0)
meta["neuron"] = meta["neuron"].astype(str)
nodes = [n for n in meta["neuron"] if n in counts.columns]
meta = meta.set_index("neuron").loc[nodes]
types = meta["type"]
counts = counts[nodes]
counts = counts.loc[counts.sum(axis=1) > 0]
print(f"{len(nodes)} neurons, {counts.shape[0]} genes")
print(types.value_counts().to_dict())
lfc = one_vs_all_log2fc(counts)          # genes x neurons, one-vs-all log2FC on CPM

edges = pd.read_csv(INPUT_DIR / "edges.csv")
conn = build_connectome(edges, types)
print(f"connectome (undirected projection): {conn.number_of_edges()} edges")

# gene sources for expression networks: all genes, plus each regulon collection
sources, reg_sets = {"all_genes": None}, {}
reg_files = {nm: read_gene_sets(p) for nm, p in REGULON_FILES.items()}
for name, sets in reg_files.items():
    sources[f"regulon_{name}"] = set().union(*sets.values())
    reg_sets.update({f"{name}:{k}": v for k, v in sets.items()})
if len(reg_files) > 1:
    sources["regulon_combined"] = set().union(*reg_sets.values())
print("gene sources:", list(sources))

126 neurons, 13546 genes
{'interneuron': 45, 'sensory': 39, 'motor': 28, 'pharyngeal': 14}
connectome (undirected projection): 1474 edges
gene sources: ['all_genes', 'regulon_TF Link']


In [7]:
# Analysis Functions
def expression_weights(lfc, X):
    """Dense symmetric weight matrix (nodes x nodes) from top-X log2FC genes."""
    M = lfc.values
    X = min(X, M.shape[0])
    top = np.zeros(M.shape, dtype=bool)
    np.put_along_axis(top, np.argsort(-M, axis=0)[:X], True, axis=0)
    L = np.where(top, M, 0.0)
    T = top.astype(float)
    W = (L.T @ T + T.T @ L) / 2.0
    np.fill_diagonal(W, 0.0)
    return W


def threshold_graph(W, nodes, types, n_edges):
    iu = np.triu_indices_from(W, k=1)
    w = W[iu]
    order = np.argsort(-w)[:n_edges]
    order = order[w[order] > 0]
    G = nx.Graph()
    G.add_nodes_from(nodes)
    for k in order:
        G.add_edge(nodes[iu[0][k]], nodes[iu[1][k]], weight=float(w[k]))
    nx.set_node_attributes(G, types.to_dict(), "type")
    return G


def _safe(f, *a, **k):
    try:
        v = f(*a, **k)
        return float(v) if np.isfinite(v) else np.nan
    except Exception:
        return np.nan


def local_assortativity(G):
    """Piraveenan-style local degree assortativity; values sum to global r."""
    deg = dict(G.degree())
    M = G.number_of_edges()
    if M == 0:
        return pd.Series(dtype=float)
    rem = {v: deg[v] - 1 for v in G}
    two_m = 2 * M
    mu = sum(rem[v] * deg[v] for v in G) / two_m
    var = sum(rem[v] ** 2 * deg[v] for v in G) / two_m - mu ** 2
    out = {}
    for v in G:
        if deg[v] == 0 or var == 0:
            out[v] = np.nan
            continue
        out[v] = rem[v] * sum(rem[u] - mu for u in G[v]) / (two_m * var)
    return pd.Series(out)


def neighbour_type_homophily(G):
    out = {}
    for v in G:
        nb = list(G[v])
        out[v] = np.mean([G.nodes[u]["type"] == G.nodes[v]["type"] for u in nb]) if nb else np.nan
    return pd.Series(out)


def assort_stats(G, n_perm, seed):
    obs_t = _safe(nx.attribute_assortativity_coefficient, G, "type")
    obs_d = _safe(nx.degree_assortativity_coefficient, G)
    M = G.number_of_edges()
    rng = random.Random(seed)
    nt, nd = [], []
    if M >= 4 and n_perm > 0:
        for _ in range(n_perm):
            H = G.copy()
            try:
                nx.double_edge_swap(H, nswap=5 * M, max_tries=100 * M, seed=rng.randrange(1 << 30))
            except Exception:
                continue
            nt.append(_safe(nx.attribute_assortativity_coefficient, H, "type"))
            nd.append(_safe(nx.degree_assortativity_coefficient, H))

    def pval(obs, null):
        null = np.array([x for x in null if np.isfinite(x)])
        if not np.isfinite(obs) or len(null) == 0:
            return np.nan
        return (1 + np.sum(np.abs(null - null.mean()) >= abs(obs - null.mean()))) / (1 + len(null))

    return dict(type_assort=obs_t, type_assort_p=pval(obs_t, nt),
                degree_assort=obs_d, degree_assort_p=pval(obs_d, nd))


def analysis1(conn, lfc, types, sources, Xs, n_perm, out, seed):
    nodes = list(conn.nodes)
    n_edges = conn.number_of_edges()
    rows, nets = [], {}

    base = assort_stats(conn, n_perm, seed)
    rows.append(dict(source="connectome", X=np.nan, n_edges=n_edges, **base))
    pd.DataFrame({"local_degree_assort": local_assortativity(conn),
                  "neighbour_type_homophily": neighbour_type_homophily(conn)}
                 ).to_csv(out / "node_assortativity_connectome.csv")

    for sname, genes in sources.items():
        sub = lfc if genes is None else lfc.loc[lfc.index.intersection(list(genes))]
        if sub.shape[0] < 5:
            print(f"  skipping source {sname}: <5 genes overlap")
            continue
        for X in Xs:
            if X >= sub.shape[0]:
                print(f"  skipping {sname} X={X}: X >= number of genes ({sub.shape[0]}), top-X is not selective")
                continue
            W = expression_weights(sub, X)
            G = threshold_graph(W, nodes, types, n_edges)
            nets[(sname, X)] = (G, W)
            st = assort_stats(G, n_perm, seed)
            rows.append(dict(source=sname, X=X, n_edges=G.number_of_edges(), **st))
            if X == Xs[len(Xs) // 2]:
                pd.DataFrame({"local_degree_assort": local_assortativity(G),
                              "neighbour_type_homophily": neighbour_type_homophily(G)}
                             ).to_csv(out / f"node_assortativity_{sname}_X{X}.csv")
        print(f"  analysis 1: {sname} done")

    res = pd.DataFrame(rows)
    res.to_csv(out / "analysis1_assortativity.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, col, title in zip(axes, ["type_assort", "degree_assort"],
                              ["Neuron-type assortativity", "Degree assortativity"]):
        for sname in sources:
            d = res[res.source == sname].sort_values("X")
            ax.plot(d.X, d[col], marker="o", label=sname)
        ax.axhline(res.loc[res.source == "connectome", col].iloc[0], color="k", ls="--",
                   label="connectome")
        ax.set_xlabel("X (top DE genes per neuron)")
        ax.set_ylabel(col)
        ax.set_title(title)
    axes[0].legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(out / "analysis1_assortativity_vs_X.png", dpi=150)
    plt.show()
    return res, nets

In [ ]:
a1, nets = analysis1(conn, lfc, types, sources, XS, N_PERM, OUT, SEED)
a1.round(3)

/opt/anaconda3/envs/biol4559/lib/python3.13/site-packages/networkx/algorithms/assortativity/correlation.py:282: RuntimeWarning: invalid value encountered in scalar divide
  r = (t - s) / (1 - s)
